In [37]:
from ultralytics import YOLO
import cv2
import os
import numpy as np
import matplotlib.pyplot as plt
from helpers import *
from shapely.geometry import Polygon
from scipy.interpolate import interp1d
from logics import *
import json
import pickle
MANUAL_RETURN_VALUE = -100000

# Load model
model = YOLO("/ssd1/tuannw/yolo/4batch3/weights/best.pt")


  

In [38]:
# Đường dẫn đến file JSON
# file_path = '/ssd1/tuannw/counting_longle/missing_case (1).json'
file_path =  "/ssd1/tuannw/processed_video/inventory_data.json"
# Đọc file JSON
with open(file_path, 'r', encoding='utf-8') as f:
    metadata = json.load(f)


In [ ]:
# Define class-specific colors
color_map = {
    0: (255, 0, 0),   # Red (in RGB)
    1: (0, 0, 255),   # Blue (in RGB)
}

PASSED_CASE = []
FAILED_CASE = []
MANUAL_CASE = []
SKIPPED_CASE = []

top_face_class_idx = 0

for top_image_path in metadata.keys():
    # if top_image_path != '/ssd1/tuannw/output_fix_merged/[30-05-25]/FOOD - Double deep/Rack BA/BA - 014 - 1_.png':
    #     continue
    
    # # TEST MANUAL CASE
    # nums_miss_gt = metadata[top_image_path]["nums_miss"]
    # if nums_miss_gt != -1:
    #     continue
    
     
    
    ground_truth_missing = metadata[top_image_path]['missing']
    nums_miss_gt = metadata[top_image_path]["nums_miss"]
    
    # # ONLY COUNT MISSING CASE
    # if ground_truth_missing == False:
    #     continue
    
    print(top_image_path)
    if not os.path.exists(top_image_path):
        continue

    front_image_path = metadata[top_image_path]['front_image']
        ### Process front image
    # Chạy inference
    results = model(front_image_path, verbose=False, agnostic_nms=True)
    result = results[0]

    boxes = result.boxes.data.cpu().numpy().tolist()
    
    # Kiểm tra xem có masks không
    if result.masks is None:
        print(f"Skipping {top_image_path} due to no masks detected")
        SKIPPED_CASE.append(top_image_path)
        continue
    masks = result.masks.xy

    # Kiểm tra xem có boxes và masks không
    if len(boxes) == 0 or len(masks) == 0:
        print(f"Skipping {top_image_path} due to no valid detections")
        SKIPPED_CASE.append(top_image_path)
        continue
    box_mask_pairs = list(zip(boxes, masks))
    box_mask_pairs_sorted = sorted(box_mask_pairs, key=lambda x: -x[0][3])
    boxes, masks = zip(*box_mask_pairs_sorted)

    boxes = np.array(boxes)
    boxes, masks = remove_overlapping_boxes(boxes, masks, ioa_thresh=0.7)
    masks = np.array([find_corners(mask) for mask in masks])

    positive_boxes = boxes[boxes[:, -1] == 1 - top_face_class_idx]
    negative_boxes = boxes[boxes[:, -1] == top_face_class_idx]
    positive_masks = [mask for box, mask in zip(boxes, masks) if int(box[-1]) == 1 - top_face_class_idx]
    negative_masks = [mask for box, mask in zip(boxes, masks) if int(box[-1]) == top_face_class_idx]


    front_img = cv2.imread(front_image_path)

    layers_front_boxes = {}
    layers_front_masks = {}
    layer_idx = 1
    for idx, bbox in enumerate(positive_boxes):
        if f"layer_{layer_idx}" not in layers_front_boxes:
            layers_front_boxes[f"layer_{layer_idx}"] = [bbox]
            layers_front_masks[f"layer_{layer_idx}"] = [positive_masks[idx]]
            continue

        y_cur = (bbox[1] + bbox[3]) / 2
        y_cluster_center = sum([(b[1] + b[3]) / 2 for b in layers_front_boxes[f"layer_{layer_idx}"]]) / len(layers_front_boxes[f"layer_{layer_idx}"])
        avg_positive_boxes_height = sum([(b[3] - b[1]) for b in layers_front_boxes[f"layer_{layer_idx}"]]) / len(layers_front_boxes[f"layer_{layer_idx}"])
        if abs(y_cur - y_cluster_center) > avg_positive_boxes_height * 0.6:
            layer_idx += 1
            layers_front_boxes[f"layer_{layer_idx}"] = [bbox]
            layers_front_masks[f"layer_{layer_idx}"] = [positive_masks[idx]]
            continue
        else:
            layers_front_boxes[f"layer_{layer_idx}"].append(bbox)
            layers_front_masks[f"layer_{layer_idx}"].append(positive_masks[idx])


    all_areas = sorted([ (box[2] - box[0]) * (box[3] - box[1]) for boxes in layers_front_boxes.values() for box in boxes ])
    global_median_area = np.median(all_areas) if len(all_areas) > 0 else 0

    layers_front_filtered_boxes = {}
    layers_front_filtered_masks = {}
    for key, boxes in layers_front_boxes.items():
        masks_in_layer = layers_front_masks[key]  # lấy mask từ layers_front_masks
        areas = [Polygon(mask).area for mask in masks_in_layer]
        if len(areas) == 0:
            layers_front_filtered_boxes[key] = []
            layers_front_filtered_masks[key] = []
            continue
        filtered_boxes = []
        filtered_masks = []
        for idx, (box, area) in enumerate(zip(boxes, areas)):
            if area >= 0.2 * global_median_area:
                filtered_boxes.append(box)
                filtered_masks.append(masks_in_layer[idx])  # lấy mask theo đúng index box
        layers_front_filtered_boxes[key] = filtered_boxes
        layers_front_filtered_masks[key] = filtered_masks

    front_merged_layers_dict = check_and_merge_layers(layers_front_filtered_boxes, layers_front_filtered_masks)
    num_layers_front = len(front_merged_layers_dict)

        ### Process top image
    # Chạy inference
    results = model(top_image_path, verbose=False, agnostic_nms=True)
    
    result = results[0]

    boxes = result.boxes.data.cpu().numpy().tolist()
    # Kiểm tra xem có masks không
    if result.masks is None:
        print(f"Skipping {top_image_path} due to no masks detected")
        SKIPPED_CASE.append(top_image_path)
        continue
    masks = result.masks.xy
    # Kiểm tra xem có boxes và masks không
    if len(boxes) == 0 or len(masks) == 0:
        print(f"Skipping {top_image_path} due to no valid detections")
        SKIPPED_CASE.append(top_image_path)
        continue
    

    box_mask_pairs = list(zip(boxes, masks))
    box_mask_pairs_sorted = sorted(box_mask_pairs, key=lambda x: -x[0][3])
    boxes, masks = zip(*box_mask_pairs_sorted)

    boxes = np.array(boxes)
    boxes, masks = remove_overlapping_boxes(boxes, masks, ioa_thresh=0.7)
    masks = np.array([find_corners(mask) for mask in masks])

    positive_boxes = boxes[boxes[:, -1] == 1 - top_face_class_idx]
    negative_boxes = boxes[boxes[:, -1] == top_face_class_idx]
    positive_masks = [mask for box, mask in zip(boxes, masks) if int(box[-1]) == 1 - top_face_class_idx]
    negative_masks = [mask for box, mask in zip(boxes, masks) if int(box[-1]) == top_face_class_idx]

    boxes_copy = boxes.copy()
    masks_copy = list(masks).copy()

    if len(front_merged_layers_dict) == 0:
        print(f"Skipping {top_image_path} due to no valid front layers")
        SKIPPED_CASE.append(top_image_path)
        continue
    num_front_boxes = len(front_merged_layers_dict[f"layer_{len(front_merged_layers_dict)}"]["boxes"])
    
    if num_front_boxes == 0:
        print(f"Skipping {top_image_path} due to no front boxes")
        SKIPPED_CASE.append(top_image_path)
        continue
    # Kiểm tra xem có đủ negative_boxes không
    if len(negative_boxes) == 0:
        print(f"Skipping {top_image_path} due to no negative boxes")
        SKIPPED_CASE.append(top_image_path)
        continue
    min_y_distance_top_face = np.min([n_box[3] - n_box[1] for n_box in negative_boxes[:num_front_boxes]])
    
    top_img = cv2.imread(top_image_path)
    
    layers_top_boxes = {}
    layers_top_masks = {}
    layer_idx = 1

    # Tính median chiều cao của positive_boxes
    positive_heights = [b[3] - b[1] for b in positive_boxes]
    positive_heights = sorted(positive_heights)
    avg_positive_boxes_height = np.median(positive_heights)


    for idx, bbox in enumerate(positive_boxes):
        box_height = bbox[3] - bbox[1]
        # Loại các box có chiều cao nhỏ hơn 0.2 * median
        if box_height < 0.2 * avg_positive_boxes_height:
            continue

        if f"layer_{layer_idx}" not in layers_top_boxes:
            layers_top_boxes[f"layer_{layer_idx}"] = [bbox]
            layers_top_masks[f"layer_{layer_idx}"] = [positive_masks[idx]]
            continue

        y_cur = (bbox[1] + bbox[3]) / 2
        y_cluster_center = sum([(b[1] + b[3]) / 2 for b in layers_top_boxes[f"layer_{layer_idx}"]]) / len(layers_top_boxes[f"layer_{layer_idx}"])

        if abs(y_cur - y_cluster_center) > avg_positive_boxes_height * 0.6:
            layer_idx += 1
            layers_top_boxes[f"layer_{layer_idx}"] = [bbox]
            layers_top_masks[f"layer_{layer_idx}"] = [positive_masks[idx]]
            continue
        else:
            layers_top_boxes[f"layer_{layer_idx}"].append(bbox)
            layers_top_masks[f"layer_{layer_idx}"].append(positive_masks[idx])


    num_layers_top = len(layers_top_boxes)


    all_areas = sorted([ (box[2] - box[0]) * (box[3] - box[1]) for boxes in layers_top_boxes.values() for box in boxes ])
    global_median_area = np.median(all_areas) if len(all_areas) > 0 else 0

    layers_top_filtered_boxes = {}
    layers_top_filtered_masks = {}
    for key, boxes in layers_top_boxes.items():
        masks_in_layer = layers_top_masks[key]  # lấy mask từ layers_top_masks
        areas = [Polygon(mask).area for mask in masks_in_layer]
        if len(areas) == 0:
            layers_top_filtered_boxes[key] = []
            layers_top_filtered_masks[key] = []
            continue
        filtered_boxes = []
        filtered_masks = []
        for idx, (box, area) in enumerate(zip(boxes, areas)):
            if area >= 0.2 * global_median_area:
                filtered_boxes.append(box)
                filtered_masks.append(masks_in_layer[idx])  # lấy mask theo đúng index box
        layers_top_filtered_boxes[key] = filtered_boxes
        layers_top_filtered_masks[key] = filtered_masks

    top_merged_layers_dict = check_and_merge_layers(layers_top_filtered_boxes, layers_top_filtered_masks)
    num_layers_top = len(top_merged_layers_dict)
    
    
    if num_layers_top == 0:
        print(f"Skipping {top_image_path} due to no valid layers in top view.")
        SKIPPED_CASE.append(top_image_path)
        continue
    ### FAIL RULE LOGIC
    last_layer_top_boxes = top_merged_layers_dict[f"layer_{num_layers_top}"]["boxes"]
    last_layer_top_masks = top_merged_layers_dict[f"layer_{num_layers_top}"]["masks"]
    
    failed_rule = []

    # Kiểm tra xem số lượng front-layer của top view và front view có khớp nhau không (rule 1)
    is_missing = False
    
    if not match_layers(front_merged_layers_dict, top_merged_layers_dict):
        is_missing = True
        failed_rule.append(100)

    # Kiểm tra xem khoảng cách 2 đường line có dưới ngưỡng cho phép không (rule 2)
    if num_layers_top > 1:
        last_layer_boxes = top_merged_layers_dict[f"layer_{num_layers_top}"]["boxes"]
        pre_last_layer_boxes = top_merged_layers_dict[f"layer_{num_layers_top - 1}"]["boxes"]

        last_layer_boxes = sorted(last_layer_boxes, key=lambda x: x[3], reverse=True)
        pre_last_layer_boxes = sorted(pre_last_layer_boxes, key=lambda x: x[3], reverse=True)

        box1 = last_layer_boxes[0]
        box2 = pre_last_layer_boxes[0]
        if abs(box1[3] - box2[1]) > min_y_distance_top_face * 0.6:
            is_missing = True
            failed_rule.append(200)

    # Kiểm tra chiều dài x phủ pallet (rule 3)
    missing_list = [info['missing'] for info in top_merged_layers_dict.values()]
    all_missing = all(info['missing'] for info in top_merged_layers_dict.values())
    if all_missing:
        is_missing = True
        failed_rule.append(300)
    # missing_list = [info['missing'] for info in top_merged_layers_dict.values()]
    # if missing_list[-1]:
    #     is_missing = True
    #     failed_rule.append(300)
    # Kiểm tra diện tích phủ của top view có đủ không (rule 4)    
    image = cv2.imread(top_image_path)
    # boxes_copy = np.array(boxes_copy)
    # masks_copy = np.array(masks_copy)
    if is_missing == False:
        is_missing = check_missing_by_area(image, boxes_copy, masks_copy, len(negative_boxes), top_face_class_idx)
        if is_missing:
            failed_rule.append(400)
    
    
    manual_check = False  
    manual_rule = []    
    
    
    # Kiểm tra cả 2 layer cuối đều thiếu thì manual (rule 5)
    if len(missing_list) >= 2 and missing_list[-1] and missing_list[-2]:
        manual_check = True
        manual_rule.append(500)



    ### Process missing logic
    positive_indices = boxes_copy[:, -1] == 1 - top_face_class_idx
    negative_indices = boxes_copy[:, -1] == top_face_class_idx

    positive_boxes_copy = boxes_copy[positive_indices]
    negative_boxes_copy = boxes_copy[negative_indices]
    positive_masks_copy = [masks_copy[idx] for idx, v in enumerate(positive_indices) if v == True]
    negative_masks_copy = [masks_copy[idx] for idx, v in enumerate(negative_indices) if v == True]

    filtered_negative_boxes, filtered_negative_masks, remove_negative_boxes, remove_negative_masks, area_list, _ = filter_negative_boxes(negative_boxes, negative_masks_copy)
    filtered_data_boxes_accept = np.concatenate((last_layer_top_boxes, filtered_negative_boxes), axis=0)
    filtered_data_boxes_accept = convert_bbox_xyxy_to_xywh(filtered_data_boxes_accept)
    filtered_data_masks_accept = last_layer_top_masks + filtered_negative_masks
    
    filtered_data_boxes_accept, filtered_data_boxes_ignore, filtered_data_masks_accept, filtered_data_masks_ignore, bx, by, num_front_face_in_line, filtered_data_class_ids_accept = interpolate_boundary(filtered_data_boxes_accept, filtered_data_masks_accept, target_class_idx=top_face_class_idx)
    filtered_data_boxes_accept = np.array(filtered_data_boxes_accept)
    filtered_data_boxes_accept = filter_data_below_front_face(filtered_data_boxes_accept, threshold=0.9)


    # Kiểm tra boxes đã lọc có đủ front và top face không (rule 6)
    num_class_neg = sum(1 for obj in filtered_data_boxes_accept if obj[5] == top_face_class_idx)
    num_class_pos = sum(1 for obj in filtered_data_boxes_accept if obj[5] == 1 - top_face_class_idx)
    if num_class_neg < num_class_pos:
        num_class_neg = num_class_pos
    if num_class_pos != num_front_face_in_line:
        num_class_neg = MANUAL_RETURN_VALUE
        
    top_face_masks_accept = [mask for mask, class_id in zip(filtered_data_masks_accept, filtered_data_class_ids_accept) if class_id == top_face_class_idx]
    front_face_masks_accept = [mask for mask, class_id in zip(filtered_data_masks_accept, filtered_data_class_ids_accept) if class_id == 1 - top_face_class_idx]
    all_top_face_masks = top_face_masks_accept + filtered_data_masks_ignore
    
    front_face_masks_accept = np.array([mask for mask, obj in zip(filtered_data_masks_accept, filtered_data_boxes_accept) if obj[5] == 1 - top_face_class_idx])
    
    
    matched_top, matched_front = find_top_for_front(front_face_masks_accept, all_top_face_masks, threshold=200)
 
    # case dư
    top_face_stack = find_all_upper_layers_by_mask(matched_top, all_top_face_masks, threshold=30)
    nums_miss = len(top_face_stack)
    if num_class_neg == MANUAL_RETURN_VALUE:
        # nums_miss = MANUAL_RETURN_VALUE
        manual_check = True
        manual_rule.append(600)
        
    # # case thiếu
    # top_face_stack = find_all_upper_layers_by_mask(top_face_masks_accept, all_top_face_masks, threshold=30)
    # nums_miss = len(top_face_stack)
    # if num_class_neg == MANUAL_RETURN_VALUE:
    #     nums_miss = MANUAL_RETURN_VALUE   
    #     manual_check = True
    #     manual_rule.append(600)
    
    # Kiểm tra góc nghiêng của các box trên top face (rule 7)
    for box in top_face_stack:
        _, _, angle = compute_box_slope(box)
        if angle > 12:
            # nums_miss = MANUAL_RETURN_VALUE
            manual_check = True
            
            manual_rule.append(700)
            break

    if manual_check:
        nums_miss = MANUAL_RETURN_VALUE    
            
    
    # TEST MANUAL CASE
    # if nums_miss != nums_miss_gt:
        
        
    
    
    # if nums_miss == MANUAL_RETURN_VALUE:
    #     if nums_miss_gt != -1:
    #         MANUAL_CASE.append([top_image_path, manual_rule])
    #     else:
    #         PASSED_CASE.append([top_image_path, manual_rule])
    # elif nums_miss != nums_miss_gt:
    #     FAILED_CASE.append([top_image_path, failed_rule + manual_rule])
    # else:
    #     PASSED_CASE.append([top_image_path, failed_rule + manual_rule])
    
    if nums_miss != nums_miss_gt:
        if nums_miss != MANUAL_RETURN_VALUE:
            FAILED_CASE.append([top_image_path, failed_rule])
        else:
            MANUAL_CASE.append([top_image_path, manual_rule])
    else:
        PASSED_CASE.append([top_image_path, []])


    print("\n", nums_miss)     

/ssd1/tuannw/processed_video/images_video_birefnet/BE-002-2_Top.png
Skipping /ssd1/tuannw/processed_video/images_video_birefnet/BE-002-2_Top.png due to no negative boxes
/ssd1/tuannw/processed_video/images_video_birefnet/BE-004-5_Top.png
Skipping /ssd1/tuannw/processed_video/images_video_birefnet/BE-004-5_Top.png due to no valid layers in top view.
/ssd1/tuannw/processed_video/images_video_birefnet/BE-006-1_Top.png
Skipping /ssd1/tuannw/processed_video/images_video_birefnet/BE-006-1_Top.png due to no masks detected
/ssd1/tuannw/processed_video/images_video_birefnet/BE-006-3_Top.png
All directions moved: [['up1', 'up1'], ['up1']]

 5
/ssd1/tuannw/processed_video/images_video_birefnet/BE-008-1_Top.png
All directions moved: [['up1'], []]

 -100000
/ssd1/tuannw/processed_video/images_video_birefnet/BE-008-5_Top.png
All directions moved: []

 -100000
/ssd1/tuannw/processed_video/images_video_birefnet/BE-010-3_Top.png
All directions moved: [['up1'], []]

 3
/ssd1/tuannw/processed_video/image

In [40]:
print(len(PASSED_CASE), "PASSED")
for case, rule in PASSED_CASE:
    print(case, rule)



16 PASSED
/ssd1/tuannw/processed_video/images_video_birefnet/BE-006-3_Top.png []
/ssd1/tuannw/processed_video/images_video_birefnet/BE-010-3_Top.png []
/ssd1/tuannw/processed_video/images_video_birefnet/BE-012-4_Top.png []
/ssd1/tuannw/processed_video/images_video_birefnet/BE-014-1_Top.png []
/ssd1/tuannw/processed_video/images_video_birefnet/BE-034-6_Top.png []
/ssd1/tuannw/processed_video/images_video_birefnet/BE-044-6_Top.png []
/ssd1/tuannw/processed_video/images_video_birefnet/BE-054-1_Top.png []
/ssd1/tuannw/processed_video/images_video_birefnet/BE-060-3_Top.png []
/ssd1/tuannw/processed_video/images_video_birefnet/BE-070-5_Top.png []
/ssd1/tuannw/processed_video/images_video_birefnet/BE-072-1_Top.png []
/ssd1/tuannw/processed_video/images_video_birefnet/BE-078-1_Top.png []
/ssd1/tuannw/processed_video/images_video_birefnet/BE-080-3_Top.png []
/ssd1/tuannw/processed_video/images_video_birefnet/BE-090-4_Top.png []
/ssd1/tuannw/processed_video/images_video_birefnet/BE-092-2_Top.png

In [41]:
# 67 chỉ có failed_rule lan
# 64 threshold 50
# 62 threshold 30

In [42]:
# fail ,manual  : slope > 12,  thresh 30
# 58 
# 49 

In [43]:
# fail ,manual  : slope > 12,  thresh 30, sửa num front in line
# 72
# 23

# 68
# 23

In [44]:
# 55
# 39 thresh num -5 5 , output mới

In [45]:
# 49  case dư  output mới
# 54

# 48  case thiếu output mới
# 55

In [46]:

# 51 case dư output cũ
# 54

# 50 case thiếu output cũ
# 55


In [47]:
# 44 case dư output mới
# 53

# 46 case thiếu output mới
# 53

In [48]:

# 49 case dư output cũ
# 53

# 49 case thiếu output cũ
# 53




In [49]:
# code a longle
# 55
# 46
# code c thaokb (-7/+0)
# 47
# 46


# 55
# 48
# fix lan + failed_rule interpolate + sort_corners + filter_negative_boxes

# 41
# 53
# fix lan + failed_rule interpolate + sort_corners + filter_negative_boxes + add smoothing mask

# 71
# 16
# tôi đã viết lại hết notebook và bỏ 1 failed_rule manual và chạy cả case -1

# 46
# 58

# 34
# 129
# case -1 giờ chạy vào fail hoặc manual

# 72
# 95
# fix filter_negative_boxes + conf 0.4 + iou 0.6
# data training

In [50]:
# test results (fail-manual-skip)
# 150
# 82
# 49
# 52
# conf 0.4 + iou 0.6

# 154
# 84
# 45
# 50
# conf 0.25 + iou 0.7

In [51]:
print(len(PASSED_CASE))
print(len(FAILED_CASE))
print(len(MANUAL_CASE))
print(len(SKIPPED_CASE))

16
19
28
36


In [52]:
print(len(FAILED_CASE), "FAILED")
for case, rule in FAILED_CASE:
    print(case, rule)

19 FAILED
/ssd1/tuannw/processed_video/images_video_birefnet/BE-014-3_Top.png []
/ssd1/tuannw/processed_video/images_video_birefnet/BE-034-1_Top.png []
/ssd1/tuannw/processed_video/images_video_birefnet/BE-036-4_Top.png [100]
/ssd1/tuannw/processed_video/images_video_birefnet/BE-040-1_Top.png []
/ssd1/tuannw/processed_video/images_video_birefnet/BE-040-3_Top.png []
/ssd1/tuannw/processed_video/images_video_birefnet/BE-048-6_Top.png [100]
/ssd1/tuannw/processed_video/images_video_birefnet/BE-050-3_Top.png [100]
/ssd1/tuannw/processed_video/images_video_birefnet/BE-052-4_Top.png [100, 300]
/ssd1/tuannw/processed_video/images_video_birefnet/BE-054-6_Top.png [100]
/ssd1/tuannw/processed_video/images_video_birefnet/BE-056-5_Top.png [100]
/ssd1/tuannw/processed_video/images_video_birefnet/BE-072-7_Top.png [100, 300]
/ssd1/tuannw/processed_video/images_video_birefnet/BE-078-4_Top.png [100, 300]
/ssd1/tuannw/processed_video/images_video_birefnet/BE-082-1_Top.png [100]
/ssd1/tuannw/processed_vi

In [53]:
print(len(MANUAL_CASE), "MANUAL")
for case, rule in MANUAL_CASE:
    print(case, rule)

28 MANUAL
/ssd1/tuannw/processed_video/images_video_birefnet/BE-008-1_Top.png [500]
/ssd1/tuannw/processed_video/images_video_birefnet/BE-008-5_Top.png [500]
/ssd1/tuannw/processed_video/images_video_birefnet/BE-012-1_Top.png [500]
/ssd1/tuannw/processed_video/images_video_birefnet/BE-014-2_Top.png [500]
/ssd1/tuannw/processed_video/images_video_birefnet/BE-016-6_Top.png [500]
/ssd1/tuannw/processed_video/images_video_birefnet/BE-020-2_Top.png [500]
/ssd1/tuannw/processed_video/images_video_birefnet/BE-022-1_Top.png [700]
/ssd1/tuannw/processed_video/images_video_birefnet/BE-024-1_Top.png [500]
/ssd1/tuannw/processed_video/images_video_birefnet/BE-026-2_Top.png [500]
/ssd1/tuannw/processed_video/images_video_birefnet/BE-028-2_Top.png [500]
/ssd1/tuannw/processed_video/images_video_birefnet/BE-030-6_Top.png [500]
/ssd1/tuannw/processed_video/images_video_birefnet/BE-038-4_Top.png [500]
/ssd1/tuannw/processed_video/images_video_birefnet/BE-038-6_Top.png [700]
/ssd1/tuannw/processed_video

In [54]:
print(len(SKIPPED_CASE), "SKIPPED")
for case in SKIPPED_CASE:
    print(case)

36 SKIPPED
/ssd1/tuannw/processed_video/images_video_birefnet/BE-002-2_Top.png
/ssd1/tuannw/processed_video/images_video_birefnet/BE-004-5_Top.png
/ssd1/tuannw/processed_video/images_video_birefnet/BE-006-1_Top.png
/ssd1/tuannw/processed_video/images_video_birefnet/BE-020-3_Top.png
/ssd1/tuannw/processed_video/images_video_birefnet/BE-022-5_Top.png
/ssd1/tuannw/processed_video/images_video_birefnet/BE-026-1_Top.png
/ssd1/tuannw/processed_video/images_video_birefnet/BE-026-5_Top.png
/ssd1/tuannw/processed_video/images_video_birefnet/BE-028-1_Top.png
/ssd1/tuannw/processed_video/images_video_birefnet/BE-030-1_Top.png
/ssd1/tuannw/processed_video/images_video_birefnet/BE-030-2_Top.png
/ssd1/tuannw/processed_video/images_video_birefnet/BE-030-3_Top.png
/ssd1/tuannw/processed_video/images_video_birefnet/BE-032-1_Top.png
/ssd1/tuannw/processed_video/images_video_birefnet/BE-032-2_Top.png
/ssd1/tuannw/processed_video/images_video_birefnet/BE-032-5_Top.png
/ssd1/tuannw/processed_video/images_v